In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from collections import Counter
import json
# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
df_multitude = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/multitude_v3_clean.csv')
print(f"MULTITuDE loaded: {df_multitude.shape}")
df_multisocial = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/multisocial_anonymized.csv')
print(f"MultiSocial loaded: {df_multisocial.shape}")

MULTITuDE loaded: (206330, 8)
MultiSocial loaded: (472097, 8)


In [ ]:
print(df_multitude.shape)
print(df_multitude.columns.tolist())
print(df_multitude['language'].unique())
print(f"\nLanguage counts:")
print(df_multitude['language'].value_counts())

print(df_multisocial.shape)
print(df_multisocial.columns.tolist())
print(df_multisocial.head(3))
print(df_multisocial['language'].unique() if 'language' in df_multisocial.columns else 'Column not found')

(206330, 8)
['Unnamed: 0', 'text', 'label', 'multi_label', 'split', 'language', 'length', 'source']
['uk' 'ru' 'hu' 'de' 'es' 'gd' 'pl' 'zh' 'en' 'ro' 'ca' 'nl' 'bg' 'pt'
 'sk' 'el' 'hr' 'ar' 'cs' 'ga' 'sl']

Language counts:
language
ar    10367
cs    10351
hu    10349
pt    10344
nl    10344
bg    10340
de    10339
en    10338
hr    10335
ro    10335
es    10334
sl    10333
sk    10331
pl    10329
el    10328
ru    10327
uk    10324
zh    10309
gd    10276
ca     5283
ga     4714
Name: count, dtype: int64
(472097, 8)
['text', 'label', 'multi_label', 'split', 'language', 'length', 'source', 'potential_noise']
                                                text  label      multi_label  \
0                        Hola, ¿qué tal? - ¿Qué tal?      1          aya-101   
1  The breathtaking picture shared by [USER] is t...      1   v5-Eagle-7B-HF   
2                            Πρέπει να πέσετε στην σ      1  opt-iml-max-30b   

   split language  length                source  potential_no

In [ ]:
TARGET_LANGUAGES = ['en', 'de', 'cs', 'es']
multitude_langs = set(df_multitude['language'].unique())
multisocial_langs = set(df_multisocial['language'].unique())
print(TARGET_LANGUAGES)
# To check weather both the dataset have the required languages
print(f"\n {all(lang in multitude_langs for lang in TARGET_LANGUAGES)}")
print(f" {all(lang in multisocial_langs for lang in TARGET_LANGUAGES)}")

for lang in TARGET_LANGUAGES:
    mult_count = len(df_multitude[df_multitude['language'] == lang])
    multi_count = len(df_multisocial[df_multisocial['language'] == lang])
    print(f" {lang}: MULTITuDE={mult_count:,} | MultiSocial={multi_count:,} | Total={mult_count+multi_count:,}")

['en', 'de', 'cs', 'es']

 True
 True
 en: MULTITuDE=10,338 | MultiSocial=50,756 | Total=61,094
 de: MULTITuDE=10,339 | MultiSocial=30,848 | Total=41,187
 cs: MULTITuDE=10,351 | MultiSocial=17,267 | Total=27,618
 es: MULTITuDE=10,334 | MultiSocial=50,781 | Total=61,115


In [ ]:
TARGET_LANGUAGES = ['en', 'de', 'cs', 'es']

df_mult_std = df_multitude.copy()
df_mult_std['dataset_source'] = 'MULTITuDE'
df_mult_std = df_mult_std[['text', 'label', 'multi_label', 'language', 'dataset_source']]

print(df_mult_std.shape)

df_multi_std = df_multisocial.copy()
df_multi_std['dataset_source'] = 'MultiSocial'
df_multi_std = df_multi_std[['text', 'label', 'multi_label', 'language', 'dataset_source']]

print(df_multi_std.shape)

df_mult_filtered = df_mult_std[df_mult_std['language'].isin(TARGET_LANGUAGES)].copy()
df_multi_filtered = df_multi_std[df_multi_std['language'].isin(TARGET_LANGUAGES)].copy()

print(df_mult_filtered.shape)
print(df_multi_filtered.shape)

print("\nLanguage distribution after applying filtering:")
print("\nMULTITuDE:")
print(df_mult_filtered['language'].value_counts())
print("\nMultiSocial:")
print(df_multi_filtered['language'].value_counts())

(206330, 5)
(472097, 5)
(41362, 5)
(149652, 5)

Language distribution after applying filtering:

MULTITuDE:
language
cs    10351
de    10339
en    10338
es    10334
Name: count, dtype: int64

MultiSocial:
language
es    50781
en    50756
de    30848
cs    17267
Name: count, dtype: int64


In [ ]:
print("MULTITuDE multi_label values:")
print(df_mult_filtered['multi_label'].value_counts())

print("\nMultiSocial multi_label values:")
print(df_multi_filtered['multi_label'].value_counts())

df_mult_filtered['binary_label'] = df_mult_filtered['multi_label'].apply(
    lambda x: 'human' if str(x).lower() == 'human' else 'machine'
)

df_multi_filtered['binary_label'] = df_multi_filtered['multi_label'].apply(
    lambda x: 'human' if str(x).lower() == 'human' else 'machine'
)

print("\nCORRECTED MULTITuDE binary labels:")
print(df_mult_filtered['binary_label'].value_counts())
print("\nBy language:")
print(pd.crosstab(df_mult_filtered['language'], df_mult_filtered['binary_label']))

print("\nCORRECTED MultiSocial binary labels:")
print(df_multi_filtered['binary_label'].value_counts())
print("\nBy language:")
print(pd.crosstab(df_multi_filtered['language'], df_multi_filtered['binary_label']))

MULTITuDE multi_label values:
multi_label
gpt-3.5-turbo-0125          5200
human                       5199
aya-101                     5198
Mistral-7B-Instruct-v0.2    5197
vicuna-13b                  5192
v5-Eagle-7B-HF              5188
opt-iml-max-30b             5133
Llama-2-70b-chat-hf         5055
Name: count, dtype: int64

MultiSocial multi_label values:
multi_label
Mistral-7B-Instruct-v0.2    19242
gemini                      19214
v5-Eagle-7B-HF              19088
vicuna-13b                  18952
gpt-3.5-turbo-0125          18635
human                       18513
aya-101                     18343
opt-iml-max-30b             17665
Name: count, dtype: int64

CORRECTED MULTITuDE binary labels:
binary_label
machine    36163
human       5199
Name: count, dtype: int64

By language:
binary_label  human  machine
language                    
cs             1300     9051
de             1300     9039
en             1299     9039
es             1300     9034

CORRECTED MultiSocial binar

In [ ]:
def analyze_text_length(df, dataset_name):
    print(dataset_name)

    df['char_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    print(f"  Character Length - Mean: {df['char_length'].mean():.1f}, Median: {df['char_length'].median():.1f}")
    print(f"  Word Count - Mean: {df['word_count'].mean():.1f}, Median: {df['word_count'].median():.1f}")
    print(f"  Min words: {df['word_count'].min()}, Max words: {df['word_count'].max()}")

    for lang in sorted(df['language'].unique()):
        lang_df = df[df['language'] == lang]
        print(f"  {lang}:")
        print(f"    Char length - Mean: {lang_df['char_length'].mean():.1f}, Median: {lang_df['char_length'].median():.1f}")
        print(f"    Word count - Mean: {lang_df['word_count'].mean():.1f}, Median: {lang_df['word_count'].median():.1f}")

    print(f"\nBy Label:")
    for label in ['human', 'machine']:
        label_df = df[df['binary_label'] == label]
        if len(label_df) > 0:
            print(f"  {label}:")
            print(f"    Char length - Mean: {label_df['char_length'].mean():.1f}, Median: {label_df['char_length'].median():.1f}")
            print(f"    Word count - Mean: {label_df['word_count'].mean():.1f}, Median: {label_df['word_count'].median():.1f}")

    return df

df_mult_filtered = analyze_text_length(df_mult_filtered.copy(), "MULTITuDE")
df_multi_filtered = analyze_text_length(df_multi_filtered.copy(), "MultiSocial")

print("\nMULTITuDE Word Count Distribution:")
print(df_mult_filtered['word_count'].describe())

print("\nMultiSocial Word Count Distribution:")
print(df_multi_filtered['word_count'].describe())

percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f"\n{'Percentile':<12} {'MULTITuDE':<15} {'MultiSocial':<15}")
print("-" * 42)
for p in percentiles:
    mult_val = df_mult_filtered['word_count'].quantile(p/100)
    multi_val = df_multi_filtered['word_count'].quantile(p/100)
    print(f"{p}th{'':<9} {mult_val:<15.1f} {multi_val:<15.1f}")

print("KEY INSIGHTS:")
mult_mean = df_mult_filtered['word_count'].mean()
multi_mean = df_multi_filtered['word_count'].mean()
print(f"• MULTITuDE avg: {mult_mean:.1f} words")
print(f"• MultiSocial avg: {multi_mean:.1f} words")
print(f"• Difference: {abs(mult_mean - multi_mean):.1f} words")
if mult_mean > multi_mean:
    print(f"• MULTITuDE texts are ~{(mult_mean/multi_mean - 1)*100:.1f}% longer (news articles vs social media)")
else:
    print(f"• MultiSocial texts are ~{(multi_mean/mult_mean - 1)*100:.1f}% longer")

MULTITuDE
  Character Length - Mean: 1085.4, Median: 1052.0
  Word Count - Mean: 168.8, Median: 161.0
  Min words: 6, Max words: 512
  cs:
    Char length - Mean: 819.8, Median: 838.0
    Word count - Mean: 126.0, Median: 127.0
  de:
    Char length - Mean: 978.8, Median: 955.0
    Word count - Mean: 138.4, Median: 134.0
  en:
    Char length - Mean: 1460.4, Median: 1470.0
    Word count - Mean: 233.1, Median: 234.0
  es:
    Char length - Mean: 1083.0, Median: 1113.5
    Word count - Mean: 177.7, Median: 182.0

By Label:
  human:
    Char length - Mean: 1009.8, Median: 987.0
    Word count - Mean: 154.8, Median: 148.0
  machine:
    Char length - Mean: 1096.3, Median: 1064.0
    Word count - Mean: 170.8, Median: 164.0
MultiSocial
  Character Length - Mean: 165.3, Median: 92.0
  Word Count - Mean: 26.2, Median: 15.0
  Min words: 3, Max words: 200
  cs:
    Char length - Mean: 99.5, Median: 58.0
    Word count - Mean: 16.8, Median: 10.0
  de:
    Char length - Mean: 186.7, Median: 116.0

In [ ]:
TARGET_LANGUAGES = ['en', 'de', 'cs', 'es']

multi_human_counts = {}
multi_machine_counts = {}

for lang in TARGET_LANGUAGES:
    multi_lang = df_multi_filtered[df_multi_filtered['language'] == lang]
    multi_human_counts[lang] = len(multi_lang[multi_lang['binary_label'] == 'human'])
    multi_machine_counts[lang] = len(multi_lang[multi_lang['binary_label'] == 'machine'])
    print(f"{lang}: Human={multi_human_counts[lang]:,}, Machine={multi_machine_counts[lang]:,}")

min_multi_human = min(multi_human_counts.values())
min_multi_machine = min(multi_machine_counts.values())

print(f"Human: {min_multi_human:,} (from {min(multi_human_counts, key=multi_human_counts.get)})")
print(f"Machine: {min_multi_machine:,} (from {min(multi_machine_counts, key=multi_machine_counts.get)})")


mult_human_avg = 1300
mult_machine_avg = 9040

final_human_per_lang = mult_human_avg + min_multi_human
final_machine_per_lang = mult_machine_avg + min_multi_machine

print(f"Human samples per language:")
print(f"  MULTITuDE: {mult_human_avg:,}")
print(f"  + MultiSocial: {min_multi_human:,}")
print(f"  = Total: {final_human_per_lang:,}")

print(f"\nMachine samples per language:")
print(f"MULTITuDE: {mult_machine_avg:,}")
print(f"+ MultiSocial: {min_multi_machine:,}")
print(f"= Total: {final_machine_per_lang:,}")

print(f"\nFinal Dataset Size:")
print(f"Per language: {final_human_per_lang + final_machine_per_lang:,} samples")
print(f"Total (4 languages): {(final_human_per_lang + final_machine_per_lang) * 4:,} samples")

en: Human=6,320, Machine=44,436
de: Human=3,766, Machine=27,082
cs: Human=2,034, Machine=15,233
es: Human=6,393, Machine=44,388
  Human: 2,034 (from cs)
  Machine: 15,233 (from cs)
Human samples per language:
  MULTITuDE: ~1,300
  + MultiSocial: 2,034
  = Total: 3,334

Machine samples per language:
  MULTITuDE: ~9,040
  + MultiSocial: 15,233
  = Total: 24,273

Final Dataset Size:
  Per language: 27,607 samples
  Total (4 languages): 110,428 samples


In [ ]:
TARGET_LANGUAGES = ['en', 'de', 'cs', 'es']

HUMAN_PER_LANG = 3334
MACHINE_PER_LANG = 3334
MACHINE_FROM_MULT = MACHINE_PER_LANG // 2
MACHINE_FROM_MULTI = MACHINE_PER_LANG // 2

print(f"Target per language:")
print(f"Human: {HUMAN_PER_LANG:,}")
print(f"Machine: {MACHINE_PER_LANG:,}")
print(f"{MACHINE_FROM_MULT:,} from MULTITuDE (50%)")
print(f"{MACHINE_FROM_MULTI:,} from MultiSocial (50%)")
print(f"\nTotal dataset: {(HUMAN_PER_LANG + MACHINE_PER_LANG) * 4:,} samples\n")

def create_final_balanced(df_mult, df_multi, languages,
                          human_per_lang, machine_from_mult, machine_from_multi):
    balanced_dfs = []

    if 'word_count' not in df_mult.columns:
        df_mult = df_mult.copy()
        df_mult['word_count'] = df_mult['text'].str.split().str.len()
    if 'word_count' not in df_multi.columns:
        df_multi = df_multi.copy()
        df_multi['word_count'] = df_multi['text'].str.split().str.len()

    MIN_MULTI_HUMAN = 2034

    for lang in languages:
        print(f"{lang}:")
        print("-" * 60)

        mult_lang = df_mult[df_mult['language'] == lang]
        multi_lang = df_multi[df_multi['language'] == lang]

        mult_human = mult_lang[mult_lang['binary_label'] == 'human'].copy()

        multi_human = multi_lang[multi_lang['binary_label'] == 'human'].copy()
        multi_human_sorted = multi_human.sort_values('word_count', ascending=False)
        multi_human_sample = multi_human_sorted.head(MIN_MULTI_HUMAN)

        final_human = pd.concat([mult_human, multi_human_sample])

        print(f"HUMAN: {len(final_human):,}")
        print(f"MULTITuDE: {len(mult_human):,} (ALL)")
        print(f"MultiSocial: {len(multi_human_sample):,} (longest)")

        mult_machine = mult_lang[mult_lang['binary_label'] == 'machine'].copy()
        mult_machine_sorted = mult_machine.sort_values('word_count', ascending=False)
        mult_machine_sample = mult_machine_sorted.head(machine_from_mult)

        multi_machine = multi_lang[multi_lang['binary_label'] == 'machine'].copy()
        multi_machine_sorted = multi_machine.sort_values('word_count', ascending=False)
        multi_machine_sample = multi_machine_sorted.head(machine_from_multi)

        final_machine = pd.concat([mult_machine_sample, multi_machine_sample])

        print(f"MACHINE: {len(final_machine):,}")
        print(f"MULTITuDE: {len(mult_machine_sample):,} (longest, avg {mult_machine_sample['word_count'].mean():.1f} words)")
        print(f"MultiSocial: {len(multi_machine_sample):,} (longest, avg {multi_machine_sample['word_count'].mean():.1f} words)")

        print(f"TOTAL: {len(final_human) + len(final_machine):,}")
        print(f"Balance: {len(final_human):,} human : {len(final_machine):,} machine\n")

        balanced_dfs.append(final_human)
        balanced_dfs.append(final_machine)

    return pd.concat(balanced_dfs, ignore_index=True)

df_balanced_final = create_final_balanced(
    df_mult_filtered,
    df_multi_filtered,
    TARGET_LANGUAGES,
    HUMAN_PER_LANG,
    MACHINE_FROM_MULT,
    MACHINE_FROM_MULTI
)

print(f"Total samples: {len(df_balanced_final):,}")

Target per language:
  Human: 3,334
  Machine: 3,334
    - 1,667 from MULTITuDE (50%)
    - 1,667 from MultiSocial (50%)

Total dataset: 26,672 samples

en:
------------------------------------------------------------
  HUMAN: 3,333
    MULTITuDE: 1,299 (ALL)
    MultiSocial: 2,034 (longest)
  MACHINE: 3,334
    MULTITuDE: 1,667 (longest, avg 349.3 words)
    MultiSocial: 1,667 (longest, avg 167.3 words)
  TOTAL: 6,667
  Balance: 3,333 human : 3,334 machine

de:
------------------------------------------------------------
  HUMAN: 3,334
    MULTITuDE: 1,300 (ALL)
    MultiSocial: 2,034 (longest)
  MACHINE: 3,334
    MULTITuDE: 1,667 (longest, avg 243.8 words)
    MultiSocial: 1,667 (longest, avg 132.2 words)
  TOTAL: 6,668
  Balance: 3,334 human : 3,334 machine

cs:
------------------------------------------------------------
  HUMAN: 3,334
    MULTITuDE: 1,300 (ALL)
    MultiSocial: 2,034 (longest)
  MACHINE: 3,334
    MULTITuDE: 1,667 (longest, avg 203.3 words)
    MultiSocial: 1,667

In [ ]:
if 'word_count' in df_balanced_final.columns:
    df_balanced_final = df_balanced_final.drop('word_count', axis=1)

df_balanced_final = df_balanced_final.sample(frac=1, random_state=42).reset_index(drop=True)

output_path = '/content/drive/MyDrive/Colab Notebooks/OurBalancedDataset.csv'
df_balanced_final.to_csv(output_path, index=False)

print(f" {output_path}")
print(f"{len(df_balanced_final):,}")

import os
if os.path.exists(output_path):
    print("File saved")

test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/OurBalancedDataset.csv')

print(test_df.shape)
print(test_df.columns.tolist())
print(test_df.head(1))

 /content/drive/MyDrive/Colab Notebooks/OurBalancedDataset.csv
26,671
File saved
(26671, 7)
['text', 'label', 'multi_label', 'language', 'dataset_source', 'binary_label', 'char_length']
                                                text  label multi_label  \
0  Wahlergebnis: 98,9 Prozent - Kramp-Karrenbauer...      0       human   

  language dataset_source binary_label  char_length  
0       de    MultiSocial        human          293  


In [ ]:

# print(f"\n {len(df_balanced_final):,}")

# print(f"\n1. Binary Label Distribution (MUST BE 50/50):")
# print(df_balanced_final['binary_label'].value_counts())
# human_count = (df_balanced_final['binary_label']=='human').sum()
# machine_count = (df_balanced_final['binary_label']=='machine').sum()
# human_pct = human_count/len(df_balanced_final)*100
# machine_pct = machine_count/len(df_balanced_final)*100
# print(f"   Human: {human_pct:.1f}% | Machine: {machine_pct:.1f}%")
# if abs(human_count - machine_count) <= 4:
#     print(f"   PERFECTLY BALANCED!")

# print(f"\nLanguage Distribution:")
# print(df_balanced_final['language'].value_counts())

# print(f"\nSource Distribution:")
# print(df_balanced_final['dataset_source'].value_counts())

# print(f"\nLanguage x Binary Label (PERFECT BALANCE):")
# lang_binary = pd.crosstab(df_balanced_final['language'], df_balanced_final['binary_label'])
# print(lang_binary)

# print(f"\nLanguage x Source:")
# lang_source = pd.crosstab(df_balanced_final['language'], df_balanced_final['dataset_source'])
# print(lang_source)

# print(f"\nBinary Label x Source (CHECK 50/50 for Machine):")
# label_source = pd.crosstab(df_balanced_final['binary_label'], df_balanced_final['dataset_source'])
# print(label_source)

# machine_data = df_balanced_final[df_balanced_final['binary_label']=='machine']
# mult_machine_count = (machine_data['dataset_source']=='MULTITuDE').sum()
# multi_machine_count = (machine_data['dataset_source']=='MultiSocial').sum()
# print(f"\nMachine samples:")
# print(f"  MULTITuDE: {mult_machine_count:,} ({mult_machine_count/len(machine_data)*100:.1f}%)")
# print(f"  MultiSocial: {multi_machine_count:,} ({multi_machine_count/len(machine_data)*100:.1f}%)")
# if abs(mult_machine_count - multi_machine_count) <= 4:
#     print(f" 50/50 BALANCE!")

# print(f"\nText Length by Source:")
# if 'word_count' not in df_balanced_final.columns:
#     df_balanced_final['word_count'] = df_balanced_final['text'].str.split().str.len()

# for source in ['MULTITuDE', 'MultiSocial']:
#     source_data = df_balanced_final[df_balanced_final['dataset_source']==source]
#     print(f"   {source}: Mean={source_data['word_count'].mean():.1f}, Median={source_data['word_count'].median():.1f}")

# print(f"\n   Overall: Mean={df_balanced_final['word_count'].mean():.1f}, Median={df_balanced_final['word_count'].median():.1f}")
# import json

# # Create metadata
# metadata = {
#     'dataset_name': 'Perfectly Balanced 4-Language MGT Detection Dataset',
#     'total_samples': 26671,
#     'balance': {
#         'human': 13335,
#         'machine': 13336,
#         'ratio': '50/50'
#     },
#     'languages': ['en', 'de', 'cs', 'es'],
#     'samples_per_language': 6668,
#     'sources': {
#         'MULTITuDE': 11867,
#         'MultiSocial': 14804
#     },
#     'machine_source_balance': {
#         'MULTITuDE': 6668,
#         'MultiSocial': 6668,
#         'ratio': '50/50'
#     },
#     'text_stats': {
#         'overall_avg_words': 138.5,
#         'MULTITuDE_avg': 219.6,
#         'MultiSocial_avg': 73.4
#     },
#     'strategy': 'ALL MULTITuDE + balanced MultiSocial fill, prioritizing longest texts',
#     'random_seed': 42,
#     'created_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
# }

# # Save metadata
# metadata_path = '/content/drive/MyDrive/Colab Notebooks/balanced_dataset_metadata.json'
# with open(metadata_path, 'w') as f:
#     json.dump(metadata, f, indent=2)

# print(f"Metadata saved: {metadata_path}")
# print(f"\nSummary:")
# print(json.dumps(metadata, indent=2))